# Session 1: the gap between P and S gives the distance
**A. Real-world problem.** An earthquake sends out two kinds of wave that travel at different
speeds. A seismograph (a ground-motion recorder) sees the fast one first, the slower one
later. The longer the wait between them, the farther away the quake was, just like counting
seconds between lightning and thunder.

**Plan.** (1) travel time = distance / speed, (2) the P-S gap grows with distance,
(3) turn a measured gap back into a distance, (4) draw a synthetic seismogram.

**Safety note.** This repo is for learning. It cannot warn you of an earthquake and
earthquakes cannot be predicted.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# typical upper-crust speeds, APPROXIMATE values used for teaching (km/s)
VP = 6.0   # P wave: push-pull, fast
VS = 3.5   # S wave: side-to-side shaking, slower

## Step 1: travel time
A wave that travels a distance d at speed v needs t = d / v. A P wave to a station 100 km away
at 6.0 km/s needs 100 / 6.0 = 16.7 s. An S wave at 3.5 km/s needs 100 / 3.5 = 28.6 s.

In [ ]:
def travel_time(d_km, v_kms):
    """Time in seconds for a wave to cover d_km kilometres at v_kms km/s."""
    return np.asarray(d_km, dtype=float) / v_kms

## Step 2: the gap grows with distance
The S wave falls further behind the longer it travels. The gap is
gap = d / vs - d / vp. It is zero at the source and grows in a straight line with distance.

In [ ]:
def sp_gap(d_km, vp=VP, vs=VS):
    """S minus P arrival time (seconds) at distance d_km."""
    return travel_time(d_km, vs) - travel_time(d_km, vp)

## Step 3: from a measured gap back to a distance
Turn the formula around: d = gap * vp * vs / (vp - vs). With vp = 6.0 and vs = 3.5 each second
of gap is worth 6.0 * 3.5 / 2.5 = 8.4 km. This is the "lightning and thunder" rule for earthquakes.

In [ ]:
def km_per_second(vp=VP, vs=VS):
    """Distance (km) corresponding to one second of S-P gap."""
    return vp * vs / (vp - vs)


def distance_from_gap(gap_s, vp=VP, vs=VS):
    """Distance (km) to the quake from the S-P gap (s)."""
    return np.asarray(gap_s, dtype=float) * km_per_second(vp, vs)

## Step 4: a synthetic seismogram
We draw a made-up record: quiet, then a P wiggle, then a bigger S wiggle. This is invented
data for learning, not a real recording.

In [ ]:
def synthetic_trace(d_km, t_end=60.0, dt=0.05, seed=1):
    """Return (t, trace) for a made-up seismogram at distance d_km."""
    rng = np.random.default_rng(seed)
    t = np.arange(0.0, t_end, dt)
    tr = 0.03 * rng.standard_normal(t.size)
    for arrival, amp, freq in ((float(travel_time(d_km, VP)), 0.5, 1.5), (float(travel_time(d_km, VS)), 1.0, 0.8)):
        k = t >= arrival
        tr[k] += amp * np.exp(-(t[k] - arrival) / 5.0) * np.sin(2 * np.pi * freq * (t[k] - arrival))
    return t, tr

## Step 5: see it
Left: the synthetic seismogram at 100 km with the P and S arrivals marked. Right: the gap
grows in a straight line with distance, so a measured gap reads off a distance.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    d = 100.0
    tp, ts = float(travel_time(d, VP)), float(travel_time(d, VS))
    print(f"distance {d:.0f} km: P at {tp:.2f} s, S at {ts:.2f} s, gap {ts - tp:.2f} s")
    print(f"each second of gap is worth {km_per_second():.1f} km; gap {ts - tp:.2f} s -> {float(distance_from_gap(ts - tp)):.1f} km")

    t, tr = synthetic_trace(d)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(t, tr, color="#264653", lw=0.9)
    ax.axvline(tp, color="#2a9d8f", ls="--")
    ax.axvline(ts, color="#d9534f", ls="--")
    ax.annotate("P arrives", (tp, 0.85), color="#2a9d8f", ha="right")
    ax.annotate("S arrives", (ts, 0.85), color="#d9534f", ha="right")
    ax.set_xlabel("time since the quake (s)")
    ax.set_ylabel("ground motion (arbitrary units)")
    ax.set_title(f"Synthetic seismogram, station 100 km away: S-P gap {ts - tp:.1f} s")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "seismogram.png", dpi=120)

    dd = np.linspace(0, 300, 61)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(dd, sp_gap(dd), color="#e76f51", label="S-P gap (vp 6.0, vs 3.5 km/s)")
    ax2.plot(dd, sp_gap(dd, 6.0, 4.0), color="#8d99ae", ls="--", label="if vs were 4.0 km/s")
    ax2.plot([d], [ts - tp], "ko")
    ax2.set_xlabel("distance to the quake (km)")
    ax2.set_ylabel("S-P gap (s)")
    ax2.set_title("The gap grows in a straight line with distance")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "gap_vs_distance.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Change the speeds in `lesson.py` to vp = 6.5 and vs = 3.7 km/s. Predict the km-per-second rule first, then print it.
2. **Stretch.** Write `gap_table(distances)` that prints the S-P gap for 20, 50, 100 and 200 km, and check that doubling the distance doubles the gap.
3. **Explain.** Explain, in your own words, why one station gives a circle and not a point.

## Assessment
- **A. Conceptual.** Why does the P wave always arrive before the S wave? What would the seismogram look like if the quake were very close to the station?
- **B. Numerical.** A station reads an S-P gap of 15 s. How far away was the quake? Check that the answer is reasonable.
- **C. Graph and simulation interpretation.** On the gap-versus-distance graph, what is the gap at 300 km? What does the slope of the line mean? What changes if the S speed rises to 4.0 km/s?
- **D. Design and optimisation challenge.** Design: a seismograph clock can be wrong by up to 0.5 s in the S-P gap you read. How much distance error does that cause, and is it acceptable for a regional map where you want to be within 10 km?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("100 km at 6.0 km/s takes 16.7 s", abs(float(L.travel_time(100.0, 6.0)) - 16.667) < 1e-3))
    tp, ts = float(L.travel_time(100.0, L.VP)), float(L.travel_time(100.0, L.VS))
    checks.append(("worked example: station at 100 km sees P at 16.67 s, S at 28.57 s, gap 11.90 s",
                   abs(tp - 16.667) < 1e-3 and abs(ts - 28.571) < 1e-3 and abs((ts - tp) - 11.905) < 1e-3))
    checks.append(("worked example: gap 11.9 s gives back 100 km (hand: 11.905 * 8.4)",
                   abs(float(L.distance_from_gap(ts - tp)) - 100.0) < 1e-9 and abs(11.905 * 8.4 - 100.0) < 0.1))
    checks.append(("one second of gap is worth 8.4 km (6.0 * 3.5 / 2.5)", abs(L.km_per_second() - 8.4) < 1e-12))
    d = np.linspace(0, 300, 7)
    back = L.distance_from_gap(L.sp_gap(d))
    checks.append(("distance -> gap -> distance round trip is exact", np.allclose(back, d, atol=1e-9)))
    checks.append(("the gap is zero at the source", float(L.sp_gap(0.0)) == 0.0))
    g = L.sp_gap(np.array([50.0, 100.0, 200.0]))
    checks.append(("gap is proportional to distance (double the distance, double the gap)",
                   abs(g[1] / g[0] - 2.0) < 1e-12 and abs(g[2] / g[1] - 2.0) < 1e-12))
    checks.append(("a faster S wave (4.0 km/s) shrinks the gap at 100 km to 8.33 s",
                   abs(float(L.sp_gap(100.0, 6.0, 4.0)) - 8.333) < 1e-3))
    t, tr = L.synthetic_trace(100.0)
    quiet = np.abs(tr[t < 10.0]).max()
    checks.append(("synthetic trace is quiet before P and larger after S",
                   quiet < 0.2 and np.abs(tr[(t > 28.57) & (t < 35)]).max() > 0.5))
    checks.append(("a 15 s gap means 126 km", abs(float(L.distance_from_gap(15.0)) - 126.0) < 1e-9))
    checks.append(("at 300 km the gap is 35.7 s", abs(float(L.sp_gap(300.0)) - 35.714) < 1e-3))
    checks.append(("a 0.5 s timing error is worth 4.2 km", abs(0.5 * L.km_per_second() - 4.2) < 1e-9))
    checks.append(("thunder: sound at 343 m/s needs 2.9 s per kilometre", abs(1000.0 / 343.0 - 2.915) < 0.01))
    checks.append(("a 9 s flash-to-bang count is about 3 km", abs(9.0 * 343.0 / 1000.0 - 3.09) < 0.01))
    checks.append(("126 km: P needs 21.0 s, S needs 36.0 s", abs(126.0 / 6.0 - 21.0) < 1e-9 and abs(126.0 / 3.5 - 36.0) < 1e-9))
    checks.append(("slope of the gap line is 0.119 s per km", abs(float(L.sp_gap(1.0)) - 0.1190) < 5e-5))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
